In [5]:
from pathlib import Path
import pyarrow.dataset as ds
import pandas as pd

# Get the main project folder from the notebooks folder
project_root = Path.cwd().parent

# Folder containing the ingested IMDb Parquet files
imdb_folder = project_root / "data" / "raw" / "imdb_reviews"

# Treat all six Parquet files as one dataset
imdb_dataset = ds.dataset(imdb_folder, format="parquet")

# Read only the movie column since that is all we need for this step
movie_table = imdb_dataset.to_table(columns=["movie"])
df_movie_table = movie_table.to_pandas()

df_movie_table.head(20)

,movie
0,After Life (2019– )
1,The Valhalla Murders (2019– )
2,Special OPS (2020– )
3,#BlackAF (2020– )
4,The Droving (2020)
5,All About Eve (1950)
6,Runaway Train (1985)
7,Iron Fist (2017–2018)
8,The Half of It (I) (2020)
9,This Is Us (2016– )


In [6]:
# Count how many unique title strings appear in the IMDb reviews
unique_title_count = df_movie_table["movie"].nunique()

print(f"Unique IMDb title strings: {unique_title_count:,}")

Unique IMDb title strings: 453,528


In [7]:
# Count how many reviews are available for each title
title_counts = (
    df_movie_table["movie"]
    .value_counts()
    .rename_axis("movie")
    .reset_index(name="review_count")
)

print(f"Titles counted: {len(title_counts):,}")

title_counts.head(20)

Titles counted: 453,528


,movie,review_count
0,Avengers: Endgame (2019),8771
1,The Shawshank Redemption (1994),8236
2,Dil Bechara (2020),7764
3,Game of Thrones: The Iron Throne (2019) Season...,7428
4,Captain Marvel (2019),7158
5,The Dark Knight (2008),6828
6,Wonder Woman 1984 (2020),6810
7,Star Wars: Episode VIII - The Last Jedi (2017),6585
8,小丑 (2019),6514
9,Mrs. Serial Killer (2020),5419


In [8]:
# See how many titles meet different review-count thresholds
thresholds = [1, 5, 10, 25, 50, 100, 250, 500, 1000]

for threshold in thresholds:
    count = (title_counts["review_count"] >= threshold).sum()
    print(f"{threshold}+ reviews: {count:,} titles")

1+ reviews: 453,528 titles
5+ reviews: 117,932 titles
10+ reviews: 68,417 titles
25+ reviews: 31,714 titles
50+ reviews: 17,147 titles
100+ reviews: 8,972 titles
250+ reviews: 3,522 titles
500+ reviews: 1,480 titles
1000+ reviews: 456 titles


In [9]:
# Total number of reviews in the dataset
total_reviews = title_counts["review_count"].sum()

# Check how many titles and reviews remain at each review-count threshold
thresholds = [1, 5, 10, 25, 50, 100, 250, 500, 1000]

for threshold in thresholds:
    qualifying_titles = title_counts[
        title_counts["review_count"] >= threshold
    ]

    title_count = len(qualifying_titles)
    review_count = qualifying_titles["review_count"].sum()
    review_percentage = review_count / total_reviews * 100

    print(
        f"{threshold}+ reviews: "
        f"{title_count:,} titles | "
        f"{review_count:,} reviews | "
        f"{review_percentage:.1f}% of all reviews"
    )

1+ reviews: 453,528 titles | 5,571,499 reviews | 100.0% of all reviews
5+ reviews: 117,932 titles | 5,035,720 reviews | 90.4% of all reviews
10+ reviews: 68,417 titles | 4,712,539 reviews | 84.6% of all reviews
25+ reviews: 31,714 titles | 4,159,051 reviews | 74.6% of all reviews
50+ reviews: 17,147 titles | 3,660,074 reviews | 65.7% of all reviews
100+ reviews: 8,972 titles | 3,094,175 reviews | 55.5% of all reviews
250+ reviews: 3,522 titles | 2,257,886 reviews | 40.5% of all reviews
500+ reviews: 1,480 titles | 1,547,888 reviews | 27.8% of all reviews
1000+ reviews: 456 titles | 843,694 reviews | 15.1% of all reviews


In [10]:
# Keep only titles with at least 100 reviews
candidate_titles = title_counts[
    title_counts["review_count"] >= 100
].copy()

print(f"Candidate titles: {len(candidate_titles):,}")

candidate_titles.head(20)

Candidate titles: 8,972


,movie,review_count
0,Avengers: Endgame (2019),8771
1,The Shawshank Redemption (1994),8236
2,Dil Bechara (2020),7764
3,Game of Thrones: The Iron Throne (2019) Season...,7428
4,Captain Marvel (2019),7158
5,The Dark Knight (2008),6828
6,Wonder Woman 1984 (2020),6810
7,Star Wars: Episode VIII - The Last Jedi (2017),6585
8,小丑 (2019),6514
9,Mrs. Serial Killer (2020),5419


In [11]:
# Flag title strings that look like individual TV episodes
candidate_titles["is_episode"] = candidate_titles["movie"].str.contains(
    r"Season \d+, Episode \d+",
    regex=True,
    na=False
)

# Flag titles that include the word Video in the year/type section
candidate_titles["is_video"] = candidate_titles["movie"].str.contains(
    r"\(\d{4} Video\)",
    regex=True,
    na=False
)

print("Episode-like titles:", candidate_titles["is_episode"].sum())
print("Video titles:", candidate_titles["is_video"].sum())

Episode-like titles: 131
Video titles: 68


In [12]:
# Look at some episode-like title strings
candidate_titles[
    candidate_titles["is_episode"]
].head(20)

,movie,review_count,is_episode,is_video
3,Game of Thrones: The Iron Throne (2019) Season...,7428,True,False
23,"Game of Thrones: The Bells (2019) Season 8, Ep...",3892,True,False
26,Game of Thrones: The Long Night (2019) Season ...,3754,True,False
31,"Supernatural: Carry On (2020) Season 15, Episo...",3624,True,False
89,Game of Thrones: The Last of the Starks (2019)...,2281,True,False
420,The Mandalorian: Chapter 16: The Rescue (2020)...,1053,True,False
907,"Attack on Titan: Hero (2019) Season 3, Episode 17",688,True,False
967,"Game of Thrones: Winterfell (2019) Season 8, E...",659,True,False
1106,"Black Mirror: Rachel, Jack and Ashley Too (201...",607,True,False
1297,Game of Thrones: A Knight of the Seven Kingdom...,543,True,False


In [13]:
# Look at some non-episode title strings
candidate_titles[
    ~candidate_titles["is_episode"]
].head(20)

,movie,review_count,is_episode,is_video
0,Avengers: Endgame (2019),8771,False,False
1,The Shawshank Redemption (1994),8236,False,False
2,Dil Bechara (2020),7764,False,False
4,Captain Marvel (2019),7158,False,False
5,The Dark Knight (2008),6828,False,False
6,Wonder Woman 1984 (2020),6810,False,False
7,Star Wars: Episode VIII - The Last Jedi (2017),6585,False,False
8,小丑 (2019),6514,False,False
9,Mrs. Serial Killer (2020),5419,False,False
10,The Lord of the Rings: The Fellowship of the R...,5357,False,False


In [14]:
# Keep titles with at least 100 reviews and remove individual episode records
tmdb_candidates = candidate_titles[
    ~candidate_titles["is_episode"]
].copy()

print(f"Candidates before removing episodes: {len(candidate_titles):,}")
print(f"Episode titles removed: {candidate_titles['is_episode'].sum():,}")
print(f"Candidates for TMDB: {len(tmdb_candidates):,}")

Candidates before removing episodes: 8,972
Episode titles removed: 131
Candidates for TMDB: 8,841


In [15]:
# Extract the first release year from the IMDb title string
tmdb_candidates["year"] = (
    tmdb_candidates["movie"]
    .str.extract(r"\((\d{4})")[0]
)

# Remove the year/type information at the end so we have a cleaner title for TMDB search
tmdb_candidates["search_title"] = (
    tmdb_candidates["movie"]
    .str.replace(
        r"\s*\(\d{4}(?:[–-]\d{4})?[–-]?\s*(?:Video)?\)\s*$",
        "",
        regex=True
    )
    .str.strip()
)

tmdb_candidates[
    ["movie", "search_title", "year", "review_count", "is_video"]
].head(30)

,movie,search_title,year,review_count,is_video
0,Avengers: Endgame (2019),Avengers: Endgame,2019,8771,False
1,The Shawshank Redemption (1994),The Shawshank Redemption,1994,8236,False
2,Dil Bechara (2020),Dil Bechara,2020,7764,False
4,Captain Marvel (2019),Captain Marvel,2019,7158,False
5,The Dark Knight (2008),The Dark Knight,2008,6828,False
6,Wonder Woman 1984 (2020),Wonder Woman 1984,2020,6810,False
7,Star Wars: Episode VIII - The Last Jedi (2017),Star Wars: Episode VIII - The Last Jedi,2017,6585,False
8,小丑 (2019),小丑,2019,6514,False
9,Mrs. Serial Killer (2020),Mrs. Serial Killer,2020,5419,False
10,The Lord of the Rings: The Fellowship of the R...,The Lord of the Rings: The Fellowship of the Ring,2001,5357,False


In [16]:
# Remove IMDb disambiguators like (I), (II), and (III) from the search title
tmdb_candidates["search_title"] = (
    tmdb_candidates["search_title"]
    .str.replace(r"\s+\([IVXLCDM]+\)$", "", regex=True)
    .str.strip()
)

In [17]:
# Save the candidate titles so the TMDB enrichment notebook can use them
candidate_output = (
    project_root
    / "data"
    / "processed"
    / "imdb_tmdb_candidates.parquet"
)

tmdb_candidates.to_parquet(candidate_output, index=False)

print(f"Saved {len(tmdb_candidates):,} TMDB candidates")
print("Output:", candidate_output)

Saved 8,841 TMDB candidates
Output: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\imdb_tmdb_candidates.parquet
